# NightRide AI - Model Training and Evaluation

This notebook demonstrates how to train and evaluate the pothole detection model using transfer learning with EfficientNet.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import timm
from PIL import Image
import os
import numpy as np
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

In [ ]:
# Dataset class for pothole detection
class PotholeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        
        # Load positive samples (potholes)
        pothole_dir = os.path.join(root_dir, 'potholes')
        if os.path.exists(pothole_dir):
            for img_name in os.listdir(pothole_dir):
                self.images.append(os.path.join(pothole_dir, img_name))
                self.labels.append(1)
        
        # Load negative samples (normal road)
        normal_dir = os.path.join(root_dir, 'normal')
        if os.path.exists(normal_dir):
            for img_name in os.listdir(normal_dir):
                self.images.append(os.path.join(normal_dir, img_name))
                self.labels.append(0)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [ ]:
# Model training function
def train_model(model, train_loader, val_loader, num_epochs=10, device='cuda'):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    model.to(device)
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_acc = 100. * correct / total
        print(f'Epoch {epoch+1}/{num_epochs}: Loss: {train_loss/len(train_loader):.4f}, Acc: {train_acc:.2f}%')
    
    return model

## Usage

1. Prepare your dataset in `datasets/potholes/` with subfolders `potholes/` and `normal/`
2. Run the training code above
3. Save the trained model to `backend/models/pothole_efficientnet.pth`